# Title
CatBoost Benchmark
## Purpose
Explain how the CatBoost benchmark is wired into the package helpers without forcing a training run.
## Inputs
`data/gold/listings_gold.xlsx`, CatBoost helpers, and the shared benchmark runner.
## Outputs
A CatBoost artifact and benchmark rows under `results/benchmarks/...` when training is enabled.
## Key Takeaways
This notebook is safe to open on a minimal environment because execution is opt-in.


In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd

def resolve_project_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / 'config.py').exists():
            return candidate
    raise FileNotFoundError('Could not find config.py in the current path ancestry')

PROJECT_ROOT = resolve_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from config import LISTINGS_GOLD, RESULTS_DIR
from elferspot_listings.modeling.catboost_model import fit_catboost_regressor, predict_catboost_eur, save_catboost_model
from elferspot_listings.modeling.train import train_baseline_models

BENCHMARK_DIR = RESULTS_DIR / 'benchmarks' / '02_catboost_benchmark'
RUN_TRAINING = False
TRAIN_CATBOOST = True


In [ ]:
gold_exists = LISTINGS_GOLD.exists()
metrics_path = BENCHMARK_DIR / 'metrics.json'
predictions_path = BENCHMARK_DIR / 'predictions.csv'
catboost_artifact_path = BENCHMARK_DIR / 'artifacts' / 'catboost.cbm'

print(f'Gold input: {LISTINGS_GOLD}')
print(f'Gold present: {gold_exists}')
print(f'Benchmark directory: {BENCHMARK_DIR}')
print('CatBoost helper imports are ready for opt-in training or artifact loading.')

if gold_exists:
    gold_preview = pd.read_excel(LISTINGS_GOLD, nrows=3)
    print('\nGold preview:')
    print(gold_preview.to_string(index=False))


In [ ]:
if RUN_TRAINING and gold_exists:
    gold_df = pd.read_excel(LISTINGS_GOLD)
    result = train_baseline_models(
        gold_df,
        BENCHMARK_DIR,
        train_catboost=TRAIN_CATBOOST,
    )
    print(f'Trained models: {sorted(result.metrics)}')
    print('CatBoost is enabled through train_baseline_models(..., train_catboost=True).')
elif metrics_path.exists():
    metrics = json.loads(metrics_path.read_text(encoding='utf-8'))
    print('Loaded existing benchmark metrics:')
    print(pd.DataFrame(metrics).T.sort_values('mae_eur').to_string())
    if predictions_path.exists():
        predictions = pd.read_csv(predictions_path)
        print('\nPrediction sample:')
        print(predictions[predictions['model_name'] == 'catboost'].head().to_string(index=False))
    if catboost_artifact_path.exists():
        print(f'\nCatBoost artifact available at: {catboost_artifact_path}')
else:
    print('No CatBoost outputs found yet. Keep RUN_TRAINING=False for review-only use, or enable it to refresh the benchmark.')
